In [1]:
import os
import pandas as pd
import numpy as np

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam


In [2]:
# Store Info

from pathlib import Path

seed = 0
np.random.seed(seed)
tf.random.set_seed(seed)




def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


store_cols = [
    "행정동코드",
    "상권업종중분류코드",
    "상권업종중분류명",
]

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw_data"

store_path = DATA_DIR / "store_info" / "상가(상권)정보_서울.csv"

store_df = read_csv_kor(store_path, usecols=store_cols)
# store_df = read_csv_kor(store_path)  # 전체 컬럼 읽기

# display(store_df.head())
store_df.head()


,상권업종중분류코드,상권업종중분류명,행정동코드
0,I210,기타 간이,11740580
1,N104,고용 알선,11170530
2,I212,비알코올,11500620
3,R104,유원지·오락,11350670
4,G217,시계·귀금속 소매,11110615


In [4]:
# Sales Info


def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


sales_cols = [
    "기준_년분기_코드",
    "행정동_코드",
    "서비스_업종_코드",
    "당월_매출_금액",
    "월요일_매출_금액",
    "화요일_매출_금액",
    "수요일_매출_금액",
    "목요일_매출_금액",
    "금요일_매출_금액",
    "토요일_매출_금액",
    "일요일_매출_금액",
    "시간대_00~06_매출_금액",
    "시간대_06~11_매출_금액",
    "시간대_11~14_매출_금액",
    "시간대_14~17_매출_금액",
    "시간대_17~21_매출_금액",
    "시간대_21~24_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액",
    "연령대_10_매출_금액",
    "연령대_20_매출_금액",
    "연령대_30_매출_금액",
    "연령대_40_매출_금액",
    "연령대_50_매출_금액",
    "연령대_60_이상_매출_금액",
]

sales_path = DATA_DIR / "sales_info" / "서울시(추정매출-행정동).csv"

sales_df = read_csv_kor(sales_path, usecols=sales_cols)
# sales_df = read_csv_kor(sales_path) # 전체 컬럼 읽기

# display(store_df.head())
sales_df.head()


,기준_년분기_코드,행정동_코드,서비스_업종_코드,당월_매출_금액,월요일_매출_금액,화요일_매출_금액,수요일_매출_금액,목요일_매출_금액,금요일_매출_금액,토요일_매출_금액,...,시간대_17~21_매출_금액,시간대_21~24_매출_금액,남성_매출_금액,여성_매출_금액,연령대_10_매출_금액,연령대_20_매출_금액,연령대_30_매출_금액,연령대_40_매출_금액,연령대_50_매출_금액,연령대_60_이상_매출_금액
0,20253,11740700,CS300043,10751618,0,1075162,3225485,1075162,2150324,3225485,...,4945744,4569438,6450971,4300647,0,0,3655550,0,4031857,3064211
1,20253,11740700,CS300036,8249940,1485258,1337594,1078706,1236916,484219,1639632,...,1049940,0,6250490,1985068,0,76708,603421,1556581,1643437,4355411
2,20253,11740700,CS300035,661900993,131389877,81770054,123617712,55580120,135020022,134523208,...,64033698,0,378222924,191416706,0,5562102,46276687,147256645,135249608,235294587
3,20253,11740700,CS300033,115789484,24534077,39113866,4569884,19009881,23975851,3471315,...,36905738,0,68777997,41101758,0,227558,16804451,22900671,40435377,29511697
4,20253,11740700,CS300031,13984669,462456,3463794,1461362,0,0,7214314,...,7214314,0,2358525,11626144,0,0,0,0,7214314,6770355


In [5]:
# Population Info


def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 에러: {last_err}")


population_cols = [
    "행정동_코드",
    "총_유동인구_수",
    "남성_유동인구_수",
    "여성_유동인구_수",
    "연령대_20_유동인구_수",
    "연령대_30_유동인구_수",
    "연령대_40_유동인구_수",
    "시간대_06_11_유동인구_수",
    "시간대_11_14_유동인구_수",
    "시간대_14_17_유동인구_수",
    "시간대_17_21_유동인구_수",
    "시간대_21_24_유동인구_수",
    "월요일_유동인구_수",
    "화요일_유동인구_수",
    "수요일_유동인구_수",
    "목요일_유동인구_수",
    "금요일_유동인구_수",
    "토요일_유동인구_수",
    "일요일_유동인구_수"
]

population_path = DATA_DIR / "population_info" / "서울시(유동인구-행정동).csv"

if not population_path.exists():
    raise FileNotFoundError(f"파일이 없습니다: {population_path}")

population_df = read_csv_kor(population_path, usecols=population_cols)
# population_df = read_csv_kor(population_path)  # 전체 컬럼 읽기

population_df.head()


,행정동_코드,총_유동인구_수,남성_유동인구_수,여성_유동인구_수,연령대_20_유동인구_수,연령대_30_유동인구_수,연령대_40_유동인구_수,시간대_06_11_유동인구_수,시간대_11_14_유동인구_수,시간대_14_17_유동인구_수,시간대_17_21_유동인구_수,시간대_21_24_유동인구_수,월요일_유동인구_수,화요일_유동인구_수,수요일_유동인구_수,목요일_유동인구_수,금요일_유동인구_수,토요일_유동인구_수,일요일_유동인구_수
0,11740700,6677641,3098637,3579004,759362,997949,1050346,1416164,759007,734474,1030669,881319,961900,954018,954472,951764,941417,938569,975503
1,11740690,30002,13846,16156,2516,4352,5649,6401,3331,3256,4665,3948,4200,4206,4238,4229,4237,4360,4532
2,11740685,18306303,8294964,10011339,2213064,2922263,2976562,3679956,2103548,2110430,3039907,2457186,2582521,2589839,2594920,2582295,2599532,2660693,2696503
3,11740660,6680704,3100063,3580641,886276,1087042,1167095,1357223,758727,747308,1070199,885242,944868,945197,948454,938577,946974,963368,993264
4,11740650,8182706,3812368,4370337,1132456,1469108,1282904,1647752,898226,917727,1347781,1090384,1148652,1138541,1148856,1140149,1148255,1215969,1242282


In [6]:
# 데이터 컬럼 모아보기

files = {
    "store": store_df,
    "sales": sales_df,
    "population": population_df
}

for name, df in files.items():
    print(f"\n📌 {name} 데이터 컬럼 ")
    print(df.columns)



📌 store 데이터 컬럼 
Index(['상권업종중분류코드', '상권업종중분류명', '행정동코드'], dtype='str')

📌 sales 데이터 컬럼 
Index(['기준_년분기_코드', '행정동_코드', '서비스_업종_코드', '당월_매출_금액', '월요일_매출_금액',
       '화요일_매출_금액', '수요일_매출_금액', '목요일_매출_금액', '금요일_매출_금액', '토요일_매출_금액',
       '일요일_매출_금액', '시간대_00~06_매출_금액', '시간대_06~11_매출_금액', '시간대_11~14_매출_금액',
       '시간대_14~17_매출_금액', '시간대_17~21_매출_금액', '시간대_21~24_매출_금액', '남성_매출_금액',
       '여성_매출_금액', '연령대_10_매출_금액', '연령대_20_매출_금액', '연령대_30_매출_금액',
       '연령대_40_매출_금액', '연령대_50_매출_금액', '연령대_60_이상_매출_금액'],
      dtype='str')

📌 population 데이터 컬럼 
Index(['행정동_코드', '총_유동인구_수', '남성_유동인구_수', '여성_유동인구_수', '연령대_20_유동인구_수',
       '연령대_30_유동인구_수', '연령대_40_유동인구_수', '시간대_06_11_유동인구_수',
       '시간대_11_14_유동인구_수', '시간대_14_17_유동인구_수', '시간대_17_21_유동인구_수',
       '시간대_21_24_유동인구_수', '월요일_유동인구_수', '화요일_유동인구_수', '수요일_유동인구_수',
       '목요일_유동인구_수', '금요일_유동인구_수', '토요일_유동인구_수', '일요일_유동인구_수'],
      dtype='str')
